# Phase 4 — Securities Analysis (Technical Indicators)

> Run top-to-bottom AFTER Phase 4 is built (restart the kernel first). Cells 1–3 are free (math + market data only); cells 4–5 make Gemini calls.

**Data fact:** NVDA is held by **CLT-002 / CLT-005** only. CLT-003 holds no individual stocks — asking it for NVDA analysis must produce 'you don't hold NVDA', not a fabricated chart.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env'
print('env loaded OK')

env loaded OK


## 1. Indicators compute real, verifiable values (Strategy + Factory)
Hand-rolled math — an all-gains series must give RSI = 100 exactly.

In [2]:
import numpy as np, pandas as pd
from app.indicators import indicator_factory

up = pd.Series(np.arange(1.0, 40.0))
print('RSI of pure uptrend :', indicator_factory.get('rsi').compute(up))

close = pd.Series(np.linspace(100, 120, 60))
print('SMA20               :', indicator_factory.get('sma_20').compute(close))
print('MACD                :', indicator_factory.get('macd').compute(close))

RSI of pure uptrend : {'indicator': 'rsi_14', 'value': 100.0, 'zone': 'overbought'}
SMA20               : {'indicator': 'sma_20', 'value': 116.78, 'price': 120.0, 'price_vs_ma_pct': 2.76, 'position': 'above'}
MACD                : {'indicator': 'macd', 'macd_line': 2.3278, 'signal_line': 2.3068, 'histogram': 0.021, 'momentum': 'bullish', 'recent_crossover': 'none'}


## 2. technical_analysis on real NVDA data — raw dict the LLM will explain

In [3]:
from app.tools.indicator_tools import technical_analysis
raw = technical_analysis('NVDA', ['rsi', 'sma_20', 'sma_50', 'macd', 'bollinger'])
print('series length:', raw['series_length'], '| as of:', raw['as_of'], '| source:', raw['source'])
for name, r in raw['indicators'].items():
    print(f'  {name:12} {r}')
print('\nsummary:', raw['summary'])

{"fn": "_ohlc_history", "age_s": 28.3, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T19:59:09.826321Z"}
series length: 124 | as of: 2026-07-10 | source: yfinance
  rsi          {'indicator': 'rsi_14', 'value': 57.01, 'zone': 'neutral'}
  sma_20       {'indicator': 'sma_20', 'value': 201.95, 'price': 210.96, 'price_vs_ma_pct': 4.46, 'position': 'above'}
  sma_50       {'indicator': 'sma_50', 'value': 209.07, 'price': 210.96, 'price_vs_ma_pct': 0.9, 'position': 'above'}
  macd         {'indicator': 'macd', 'macd_line': -1.7027, 'signal_line': -2.9369, 'histogram': 1.2342, 'momentum': 'bullish', 'recent_crossover': 'none'}
  bollinger    {'indicator': 'bollinger_20', 'upper': 214.11, 'middle': 201.95, 'lower': 189.79, 'price': 210.96, 'percent_b': 0.871, 'bandwidth_pct': 12.04}

summary: RSI 57.01 → neutral; price above sma_20 (+4.46%); price above sma_50 (+0.9%); MACD momentum bullish; %B 0.871 within bands


## 3. Guard rails without any LLM: cash rejected, SMA200-on-6mo unavailable (not NaN)

In [4]:
print(technical_analysis('CASH', ['rsi']))
partial = technical_analysis('NVDA', ['sma_200'])   # 6mo has ~126 points < 200
print(partial['indicators']['sma_200'])

{'symbol': 'CASH', 'status': 'not_applicable', 'message': 'N/A — cash position: cash has no price series to analyse.'}
{"fn": "_ohlc_history", "age_s": 55.0, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T19:59:36.523646Z"}
{'indicator': 'sma_200', 'unavailable': 'sma_200 needs ≥200 data points, got 124 — request a longer period'}


## 4. The agent explains the numbers (Gemini) — CLT-005 holds NVDA

In [5]:
from app.agents.securities_analysis import SecuritiesAnalysisAgent
sa = SecuritiesAnalysisAgent()
print(sa.answer(client_id='CLT-005',
    query='Perform a technical analysis of my NVIDIA position including moving averages and RSI indicators'))

{"query": "Perform a technical analysis of my NVIDIA position including moving averages and RSI indicators", "event": "agent_start", "agent": "securities_analysis", "session_id": "adhoc", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-12T19:59:52.448186Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"rows": 84, "clients": 10, "event": "portfolios_loaded", "agent": "securities_analysis", "session_id": "adhoc", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-12T20:00:00.982728Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"method": "get_price_history", "adapter": "finnhub", "error": "[finnhub] price history requires a paid plan", "event": "chain_adapter_failed", "agent": "securities_analysis", "session_id": "adhoc", "client_id": "CLT-005", "level": "warning", "timestamp": "2026-07-12T20:00:08.151083Z"}
{"method": "get_price_history", "adapter": "alpha_vantage", "error": "[alpha_vantage] period '6mo' unsupported on free tier", "event": "chain_adapter_failed", "agent": "securities_analysis", "session_id": "adhoc", "client_id": "CLT-005", "level": "warning", "timestamp": "2026-07-12T20:00:08.153064Z"}
{"method": "get_price_history", "source": "yfinance", "event": "chain_served", "agent": "securities_analysis", "session_id": "adhoc", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-12T20:00:08.663240Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"seconds": 23.92, "new_messages": 5, "tool_calls": 2, "event": "agent_done", "agent": "securities_analysis", "session_id": "adhoc", "client_id": "CLT-005", "level": "info", "timestamp": "2026-07-12T20:00:16.371908Z"}
As of July 10, 2026, your NVIDIA (NVDA) position is currently priced at $210.96.

Here is the technical analysis based on the requested indicators:

*   **RSI (14-day):** The value is 57.01, which indicates a neutral market condition.
*   **SMA 50:** The current price of $210.96 is 0.9% above the 50-day Simple Moving Average of $209.07, suggesting a slight upward trend relative to the medium-term average.
*   **SMA 200:** This indicator is currently unavailable because the 6-month data period only contains 124 data points; a minimum of 200 data points is required for this calculation.

**Overall Read:** The stock is currently in a neutral RSI zone and trading slightly above its 50-day moving average.


## 5. CLT-003 (funds only) asking about NVDA → honest 'not held'

In [6]:
print(sa.answer(client_id='CLT-003', query='Technical analysis of my NVDA position'))

{"query": "Technical analysis of my NVDA position", "event": "agent_start", "agent": "securities_analysis", "session_id": "adhoc", "client_id": "CLT-003", "level": "info", "timestamp": "2026-07-12T20:00:29.671613Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"seconds": 8.25, "new_messages": 3, "tool_calls": 1, "event": "agent_done", "agent": "securities_analysis", "session_id": "adhoc", "client_id": "CLT-003", "level": "info", "timestamp": "2026-07-12T20:00:37.927300Z"}
You do not hold a position in NVDA. As per our firm's policy, I cannot perform technical analysis on a security that is not in your portfolio.


## ✅ Acceptance check

In [7]:
assert indicator_factory.get('rsi').compute(pd.Series(np.arange(1.0, 40.0)))['value'] == 100.0
assert 0 <= raw['indicators']['rsi']['value'] <= 100
assert 'unavailable' in partial['indicators']['sma_200']
print('Phase 4 acceptance: PASS (agent answers above should cite these exact numbers)')

Phase 4 acceptance: PASS (agent answers above should cite these exact numbers)
